# Kaggle Memory-Safe Replay Analyzer
This script uses Regex instead of `json.load()` to parse the massive Kaggle replays. This ensures that you can scan millions of replays using only a few Megabytes of RAM, completely avoiding Kaggle OOM crashes.

In [ ]:
import os
import re
import gc
import pandas as pd
from pathlib import Path
from collections import defaultdict

def analyze_kaggle_datasets():
    # Find all mounted datasets
    input_dir = Path("/kaggle/input")
    replays = list(input_dir.rglob("*.json"))
    print(f"Found {len(replays)} total replays mounted on Kaggle.\n")
    
    # Dictionary to track stats
    stats = defaultdict(lambda: {"1v1_wins": 0, "4v4_wins": 0, "matches_played": 0})
    
    # Pre-compile regex for maximum speed and minimum RAM
    team_name_regex = re.compile(r'"TeamNames":\s*\[(.*?)\]')
    rewards_regex = re.compile(r'"rewards":\s*\[(.*?)\]')
    
    for i, file_path in enumerate(replays):
        if (i+1) % 5000 == 0:
            print(f"Scanned {i+1} / {len(replays)} replays...")
            gc.collect() # Force garbage collection
            
        try:
            with open(file_path, "r", encoding="utf-8") as f:
                content = f.read()
                
            teams_match = team_name_regex.search(content)
            rewards_match = rewards_regex.search(content)
            
            if not teams_match or not rewards_match:
                continue
                
            # Parse names
            teams_str = teams_match.group(1)
            team_names = [name.strip().strip('"') for name in teams_str.split(',')]
            
            # Parse rewards
            rewards_str = rewards_match.group(1)
            rewards = []
            for r in rewards_str.split(','):
                try:
                    rewards.append(float(r.strip() or 0))
                except ValueError:
                    rewards.append(0.0)
                    
            if not team_names or len(team_names) != len(rewards):
                continue
                
            num_players = len(team_names)
            max_r = max(rewards)
            winner_idx = rewards.index(max_r)
            winner_name = team_names[winner_idx]
            
            # Update stats
            for name in team_names:
                stats[name]["matches_played"] += 1
                
            if num_players == 2:
                stats[winner_name]["1v1_wins"] += 1
            elif num_players >= 3:
                stats[winner_name]["4v4_wins"] += 1
                
        except Exception as e:
            continue
            
    # Convert to Pandas DataFrame
    df_list = []
    for player, data in stats.items():
        total_wins = data["1v1_wins"] + data["4v4_wins"]
        if data["matches_played"] > 0:
            win_rate = (total_wins / data["matches_played"]) * 100
        else:
            win_rate = 0
            
        df_list.append({
            "Player Name": player,
            "1v1 Wins": data["1v1_wins"],
            "4v4 Wins": data["4v4_wins"],
            "Total Wins": total_wins,
            "Matches Played": data["matches_played"],
            "Win Rate (%)": round(win_rate, 2)
        })
        
    df = pd.DataFrame(df_list)
    df = df.sort_values(by="Win Rate (%)", ascending=False).reset_index(drop=True)
    return df

# Run analysis
df_leaderboard = analyze_kaggle_datasets()
display(df_leaderboard.head(50))
df_leaderboard.to_csv("/kaggle/working/combined_leaderboard.csv", index=False)
print("Saved complete table to /kaggle/working/combined_leaderboard.csv")
